# SmartBite YOLO26s Grocery Product Training (Colab, Drive Dataset)

This notebook assumes your Grocery_products dataset has already been converted to YOLO format and zipped on Google Drive.

Workflow:
1. Mount Drive
2. Point to YOLO dataset zip + `yolo26s.pt`
3. Install dependencies
4. Train with `subprocess.Popen` (live logs)
5. Validate and copy artifacts to Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


In [ ]:
import shlex
import shutil
import subprocess
import sys
from pathlib import Path


def run_live(cmd, env=None, cwd=None):
    print('>>', shlex.join(cmd))
    proc = subprocess.Popen(
        cmd,
        env=env,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)


## Config
Update only these paths/values if needed.


In [ ]:
# Zip in Drive (source of truth)
DATASET_ZIP_DRIVE = Path('/content/drive/My Drive/sb-colab/grocery-products-yolo.zip')

# Always unzip into a fresh local area to avoid stale/partial Drive folders
UNZIP_ROOT = Path('/content/dataset_unzip')

# Pretrained YOLO model on Drive
YOLO26S_PT_DRIVE = Path('/content/drive/My Drive/sb-colab/yolo26s.pt')

# Training config
EPOCHS = 100
IMGSZ = 640
BATCH = 16
DEVICE = '0'  # set 'cpu' if no GPU

# Output
RUNS_PROJECT = Path('/content/output')
RUN_NAME = 'smartbite_yolo_grocery_products'
BACKUP_DIR_DRIVE = Path('/content/drive/My Drive/sb-colab/models/smartbite_yolo_grocery_products')

# Optional: copy resolved dataset to a stable local path used by training
COPY_DATASET_TO_LOCAL = True
LOCAL_DATASET_ROOT = Path('/content/data/grocery-products-yolo')


In [ ]:
assert DATASET_ZIP_DRIVE.exists(), f'Missing dataset zip: {DATASET_ZIP_DRIVE}'
assert YOLO26S_PT_DRIVE.exists(), f'Missing model file: {YOLO26S_PT_DRIVE}'

# Fresh unzip every run to avoid stale folders from previous attempts
if UNZIP_ROOT.exists():
    shutil.rmtree(UNZIP_ROOT)
UNZIP_ROOT.mkdir(parents=True, exist_ok=True)

run_live([
    'unzip', '-q', '-o',
    str(DATASET_ZIP_DRIVE),
    '-d', str(UNZIP_ROOT),
])

# Resolve real dataset root robustly (handles nested zip folders)
def _looks_like_yolo_root(root: Path) -> bool:
    needed = [
        root / 'dataset.yaml',
        root / 'images' / 'train',
        root / 'images' / 'val',
        root / 'images' / 'test',
        root / 'labels' / 'train',
        root / 'labels' / 'val',
        root / 'labels' / 'test',
    ]
    return all(p.exists() for p in needed)

candidates = []
for y in UNZIP_ROOT.rglob('dataset.yaml'):
    candidates.append(y.parent)

resolved = None
seen = set()
for c in candidates:
    c = c.resolve()
    if c in seen:
        continue
    seen.add(c)
    if _looks_like_yolo_root(c):
        resolved = c
        break

assert resolved is not None, (
    'Could not locate a valid YOLO dataset root inside zip. '
    'Expected dataset.yaml + images/train,val,test + labels/train,val,test.'
)

DATASET_ROOT = resolved
DATASET_YAML = DATASET_ROOT / 'dataset.yaml'

if COPY_DATASET_TO_LOCAL:
    if LOCAL_DATASET_ROOT.exists():
        shutil.rmtree(LOCAL_DATASET_ROOT)
    LOCAL_DATASET_ROOT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DATASET_ROOT, LOCAL_DATASET_ROOT)
    DATASET_YAML = LOCAL_DATASET_ROOT / 'dataset.yaml'

# Fix dataset.yaml path portability
import yaml
cfg = yaml.safe_load(DATASET_YAML.read_text())
cfg['path'] = str(DATASET_YAML.parent)
DATASET_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False))
print('Patched dataset.yaml path ->', cfg['path'])

# Sanity-check counts so we fail early with clear info
img_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def count_imgs(d: Path) -> int:
    return sum(1 for f in d.rglob('*') if f.suffix.lower() in img_exts)

def count_labels(d: Path) -> int:
    return sum(1 for f in d.rglob('*.txt'))

train_img_n = count_imgs(DATASET_YAML.parent / 'images' / 'train')
val_img_n = count_imgs(DATASET_YAML.parent / 'images' / 'val')
test_img_n = count_imgs(DATASET_YAML.parent / 'images' / 'test')
train_lbl_n = count_labels(DATASET_YAML.parent / 'labels' / 'train')
val_lbl_n = count_labels(DATASET_YAML.parent / 'labels' / 'val')
test_lbl_n = count_labels(DATASET_YAML.parent / 'labels' / 'test')

print('train images:', train_img_n, '| train labels:', train_lbl_n)
print('val images  :', val_img_n, '| val labels  :', val_lbl_n)
print('test images :', test_img_n, '| test labels :', test_lbl_n)

assert train_img_n == 412 and train_lbl_n == 412, 'Unexpected train image/label counts.'
assert val_img_n == 133 and val_lbl_n == 133, 'Unexpected val image/label counts.'
assert test_img_n == 133 and test_lbl_n == 133, 'Unexpected test image/label counts.'

RUNS_PROJECT.mkdir(parents=True, exist_ok=True)
print('DATASET_ZIP  =', DATASET_ZIP_DRIVE)
print('DATASET_ROOT =', DATASET_ROOT)
print('DATASET_YAML =', DATASET_YAML)
print('YOLO26S_PT   =', YOLO26S_PT_DRIVE)
print('RUN DIR      =', RUNS_PROJECT / RUN_NAME)


In [ ]:
run_live([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'ultralytics', 'pyyaml'])
run_live(['nvidia-smi'])


## Train (Popen)


In [ ]:
train_script = f'''
from ultralytics import YOLO

model = YOLO(r'{YOLO26S_PT_DRIVE}')
model.train(
    data=r'{DATASET_YAML}',
    epochs={EPOCHS},
    imgsz={IMGSZ},
    batch={BATCH},
    device=r'{DEVICE}',
    project=r'{RUNS_PROJECT}',
    name=r'{RUN_NAME}',
    workers=2,
    cache=False,
)
print('Training finished.')
'''

run_live([sys.executable, '-c', train_script])


## Validate Best Checkpoint (Popen)


In [ ]:
best_pt = RUNS_PROJECT / RUN_NAME / 'weights' / 'best.pt'
assert best_pt.exists(), f'Missing best checkpoint: {best_pt}'

val_script = f'''
from ultralytics import YOLO

model = YOLO(r'{best_pt}')
metrics = model.val(data=r'{DATASET_YAML}', imgsz={IMGSZ}, batch={BATCH}, device=r'{DEVICE}')
print(metrics)
'''

run_live([sys.executable, '-c', val_script])


## Backup Artifacts to Drive


In [ ]:
src = RUNS_PROJECT / RUN_NAME
dst = BACKUP_DIR_DRIVE
assert src.exists(), f'Missing run dir: {src}'
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(src, dst)
print('Saved artifacts to:', dst)


## Notes
- This notebook expects `grocery-products-yolo.zip` in `sb-colab` and auto-unzips it when needed.
- It uses `subprocess.Popen` for install/train/val execution with live output.
- Keep `dataset.yaml` paths consistent with the dataset root.
- The dataset is single-class generic product detection: `0: product`.
